# ADHD Classification from EEG Sensor Readings using MNE and Scikit-Learn

This notebook implements an ADHD diagnostic classifier based on sliding-window epoching and Power Spectral Density (PSD) relative band power feature extraction, following the MNE architecture in `experiments/mne_examples/sleep_stage_classification.ipynb`.

### Pipeline Structure:
1. **Raw Data**: 121 subjects (61 ADHD, 60 Control) with 19 standard 10-20 EEG channels at 128Hz.
2. **Sliding-Window Epoching**: Segment continuous EEG into 15.0s windows with 0s overlap.
3. **Power-Spectral Density Features**: Estimate PSD from 15s segments (Epochs) and log-transform them.
4. **Classifiers**: `RandomForestClassifier` and `LogisticRegression`.

In [1]:
!module load CUDA/12.6.0
%matplotlib inline
from collections import defaultdict
from pprint import pprint
import os, sys
from pathlib import Path
from pprint import pprint

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mne
import torch
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegressionCV

from t_regs.models.regression.tucker_nn_regressor import TuckerRegressor
from t_regs.models.regression.tucker_nn_bcd import TuckerRegressorBCD
from t_regs.solvers.manifold.rada import RADA_RGD

import srt_gmlm_icassp_exps
from srt_gmlm_icassp_exps.config import PROJ_ROOT
from srt_gmlm_icassp_exps.eeg_adhd.create_log_psd_samples import (
    load_train_val_test_split_subjects_process
)
from srt_gmlm_icassp_exps.eeg_adhd.study_and_classification_metrics import (
    compute_classification_metrics,
    fused_lasso_matrix_torch,
    sparse_eye
)
from srt_gmlm_icassp_exps.eeg_adhd.create_log_psd_samples import (
    standardize,
    eeg_log_psd,
    load_train_val_test_split_subjects_process
)


eeg_dset_path = PROJ_ROOT / 'data' / 'eeg_adhd' / 'adhdata.csv'
fif_dir = PROJ_ROOT / 'data' / 'eeg_adhd' / 'raw_fif'

/mnt/research/ATD/SRT-GMLM/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_sample_per_subj = 4
data = load_train_val_test_split_subjects_process(
    4, fif_dir, test_ratio=0.25, val_ratio=0.25, seed=42
    )

print(f"Sample Covariate tensor shape: {data['X_train'].shape[1:]}")
print(f"X_train shape: ", data['X_train'].shape)
print(f"X_val shape: ", data['X_val'].shape)
print(f"X_test shape: ", data['X_test'].shape)
print(f"# Training samples: {data['X_train'].shape[0]}, # Validation samples: {data['X_val'].shape[0]}, # Test samples: {data['X_test'].shape[0]}")

Sample Covariate tensor shape: (19, 893)
X_train shape:  (268, 19, 893)
X_val shape:  (92, 19, 893)
X_test shape:  (124, 19, 893)
# Training samples: 268, # Validation samples: 92, # Test samples: 124


In [3]:
X = np.concat([data['X_train'],data['X_val']], axis=0)
y = np.concat([data['y_train'],data['y_val']], axis=0)
X_test = data['X_test']
y_test = data['y_test']

X, mu_train, std_train = standardize(X)
X_test,_,_ = standardize(X_test)

X_flat = X.reshape(X.shape[0],-1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

rf = RandomForestClassifier(
            random_state=42
        )
rf.fit(X_flat, y)

preds = rf.predict(X_test_flat)
probs = rf.predict_proba(X_test_flat)[:,1]
metrics = compute_classification_metrics(y_test, probs)

pprint(metrics)

/mnt/research/ATD/SRT-GMLM/.venv/lib/python3.13/site-packages/sklearn/base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


{'accuracy': 0.6935483870967742,
 'au_prc': 0.723624749638954,
 'au_roc': 0.7286458333333334,
 'f1': 0.7164179104477612}


In [3]:
study = optuna.load_study(
    study_name='v0_eegadhd_n4(2, 2)__nsps442',
    storage='sqlite:///eeg_adhd_hp_search_optuna.db'
    )

# bt_idx = np.argmin([trial.values[1] for trial in study.best_trials])
bt_idx = np.argmax([trial.values[0] for trial in study.best_trials])
best_trial = study.best_trials[bt_idx]
pprint(best_trial.params)

{'lda1': 0.005623413251903491, 'lda2': 0.0, 'tau': 0.0001}


In [ ]:
tau = best_trial.params['tau']
feature_ranks = (2,2) 
ldas = (best_trial.params['lda1'], best_trial.params['lda2']) #0.01 
fused = False
device = 'cuda:3'

X_train,mu_train,std_train = standardize(data['X_train'])
X_val,_,_ = standardize(data['X_val']) # , mu_train, std_train
X_test,_,_ = standardize(data['X_test']) # , mu_train, std_train

X_train = torch.from_numpy(X_train).to(device) #  # 
y_train = torch.from_numpy(data['y_train']).to(device) #  # 
X_val = torch.from_numpy(X_val).to(device) #  # 
y_val = torch.from_numpy(data['y_val']).to(device) #  # 
X_test = torch.from_numpy(X_test).to(device) # 
y_test = torch.from_numpy(data['y_test']).to(device) # 
dtype = torch.float64

tregs = TuckerRegressor(
    'logistic',
    feature_dims=X_train.shape[1:],
    feature_ranks=feature_ranks,
    force_lr_features=False,
    task_dims=(1,),
    ).to(device=device, dtype=dtype)

model = TuckerRegressorBCD(
    tregs, tau=tau, ldas=ldas,
    thetas=(0,0),
    device=device, Ls=[None,None], Ds=None,
    min_gradient_norm=1e-3,
    init_with_hosvd_of_grad=False,
    init_with_lbfgs_full_glm=True,
    fit_intercept=True,
    lbfgs_config={
        'max_iter': 500,
        'tolerance_grad': 1e-8,
        'tolerance_change': 1e-9,
        'line_search_fn': 'strong_wolfe'
        },
    max_it=10,
    verbosity=2,
    )

model.fit(X_train, y_train, X_val=X_val, y_val=y_val)
prob_tr = model.predict(X_train)
# train_accs[i] = float(model.score(prob_tr, y_tr))
prob_val = model.predict(X_val)
# val_accs[i] = float(model.score(prob_val, y_val))
metrics = compute_classification_metrics(y_val, prob_val)
# results.append(metrics)

preds = model.predict(X_test)
# probs = model.predict_proba(X_test)[:,1]
metric2s = compute_classification_metrics(y_test, preds)

pprint(metric2s)

Initializing with HoSVD Truncation of GLM LBFGS solution
Starting BCD_RGD Solver for Tucker Regression
Hyper-parameters:
{'coefficient_dims': (1, 19, 893),
 'feature_dims': (19, 893),
 'feature_ranks': (2, 2),
 'ldas': (0.005623413251903491, 0.0),
 'ranks': (1, 2, 2),
 'regression_type': 'logistic',
 'task_dims': (1,),
 'task_ranks': (1,),
 'tau': 0.0001,
 'thetas': (0, 0)}
Iteration    Train score    Val score    Val loss      Cost                Gradient norm    U_1 grad norm    U_2 grad norm    C grad norm    
---------    -----------    ---------    ----------    ----------------    -------------    -------------    -------------    -----------    
 1           0.9515         0.7391       1.0591e+00    1.6730879550e-01    2.9098e-01       2.3932e-01       2.9098e-01       1.5435e-05     
 2           1.0000         0.7717       2.6667e+00    3.8424196875e-02    2.2087e-02       2.2087e-02       1.4659e-02       1.4306e-06     
 3           1.0000         0.8043       1.5930e+00    